# AI Trading System Phase 2: Quant Models Evaluation
**Objective:** Evaluate default parameters for tree-based models and Logistic Regression for the Soft Voting mechanism of the Quant Agent.

In [2]:
!pip install vnstock ta catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.2 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=f0d5009031e6d0c1ff5c3e13e1932f3dcf93b12c8ec7016bd26f851ff8b13547
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [4]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.4 MB/s eta 0:00:00


In [7]:
import time
import numpy as np
import pandas as pd
from vnstock import Vnstock
import vnstock
import ta
import warnings
warnings.filterwarnings('ignore')

# Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

# Evaluation & Validation
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_score, f1_score, average_precision_score
from sklearn.preprocessing import StandardScaler

## 1. Data Preparation
Fetch 5-minute OHLCV historical data for the `VN30F1M` derivative symbol over the last 6 months using `vnstock` with source `'VCI'`.

In [9]:
# Fetch 6 months of data
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
start_date = (pd.Timestamp.today() - pd.DateOffset(months=6)).strftime('%Y-%m-%d')

print(f"Fetching data from {start_date} to {end_date}...")

stock = Vnstock().stock(symbol='VN30F1M', source='VCI')
df = stock.quote.history(start=start_date, end=end_date, interval='5m')

# Memory requirement check: use 'time' or 'tradingDate' column to parse valid datetime
if 'time' in df.columns:
    df['time'] = pd.to_datetime(df['time'])
    df.set_index('time', inplace=True)
elif 'tradingDate' in df.columns:
    df['tradingDate'] = pd.to_datetime(df['tradingDate'])
    df.set_index('tradingDate', inplace=True)

# Rename columns to standard lowercase
df.columns = [c.lower() for c in df.columns]
print(f"Data fetched: {df.shape}")
df.head()

2026-03-22 16:09:03 - vnstock.common.data - INFO - Not a stock. Company and finance data unavailable.
INFO:vnstock.common.data:Not a stock. Company and finance data unavailable.


Fetching data from 2025-09-22 to 2026-03-22...

📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive
Data fetched: (59108, 5)


,open,high,low,close,volume
time,,,,,
2025-08-27 09:10:00,1853.1,1853.4,1850.3,1851.6,2170
2025-08-27 09:15:00,1851.6,1854.9,1847.7,1849.0,8773
2025-08-27 09:20:00,1849.0,1854.5,1846.0,1853.6,9959
2025-08-27 09:25:00,1853.6,1856.5,1852.3,1854.0,7903
2025-08-27 09:30:00,1853.8,1857.9,1852.5,1856.9,5701


## 2. Feature Engineering
Calculate technical indicators: MACD, RSI(14), VWAP, and Bollinger Bands. Then, drop NaN values.

In [ ]:
# Calculate RSI(14)
df['rsi_14'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

# Calculate MACD
macd = ta.trend.MACD(close=df['close'])
df['macd'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_diff'] = macd.macd_diff()

# Calculate Bollinger Bands
bb = ta.volatility.BollingerBands(close=df['close'], window=20, window_dev=2)
df['bb_mavg'] = bb.bollinger_mavg()
df['bb_high'] = bb.bollinger_hband()
df['bb_low'] = bb.bollinger_lband()

# Calculate VWAP
df['vwap'] = ta.volume.VolumeWeightedAveragePrice(
    high=df['high'], low=df['low'], close=df['close'], volume=df['volume']
).volume_weighted_average_price()

# Drop rows with NaN values resulting from indicator lookback periods
df.dropna(inplace=True)
print(f"Data shape after feature engineering and dropping NaNs: {df.shape}")

## 3. Labeling (Target Variable Generation)
Define 3 classes based on the next 5-minute candle's price change:
- `LONG` (1): Price increases by more than 1.0 point.
- `SHORT` (-1): Price decreases by more than 1.0 point.
- `HOLD` (0): Price changes by 1.0 point or less in either direction.

In [ ]:
THRESHOLD = 1.0

# Calculate difference to next candle's close
df['next_close'] = df['close'].shift(-1)
df['price_change'] = df['next_close'] - df['close']

# Drop the last row as we don't know its future
df.dropna(inplace=True)

def assign_label(change):
    if change > THRESHOLD:
        return 1
    elif change < -THRESHOLD:
        return -1
    else:
        return 0

df['target'] = df['price_change'].apply(assign_label)

print("Class distribution:")
print(df['target'].value_counts(normalize=True))

## 4. Modeling & Evaluation
Train models using `TimeSeriesSplit` cross-validation with default parameters. Compare their precision, Macro F1-Score, PR-AUC, and Inference Time (ms).

In [ ]:
# Prepare features (X) and target (y)
features = [
    'open', 'high', 'low', 'close', 'volume',
    'rsi_14', 'macd', 'macd_signal', 'macd_diff',
    'bb_mavg', 'bb_high', 'bb_low', 'vwap'
]
X = df[features]
y = df['target']

# Define models with default parameters
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees': ExtraTreesClassifier(random_state=42),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss'),
    'CatBoost': CatBoostClassifier(random_state=42, verbose=0)
}

# Transform target to [0, 1, 2] since XGBoost expects classes starting from 0
# Original: -1 (SHORT), 0 (HOLD), 1 (LONG)
# Mapped: 0 (SHORT), 1 (HOLD), 2 (LONG)
y_mapped = y.map({-1: 0, 0: 1, 1: 2})

tscv = TimeSeriesSplit(n_splits=5)
results = []

for name, model in models.items():
    print(f"Evaluating {name}...")

    precisions, f1s, praucs, inf_times = [], [], [], []

    for train_index, test_index in tscv.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y_mapped.iloc[train_index], y_mapped.iloc[test_index]

        # Scaling is needed for LogisticRegression
        if name == 'LogisticRegression':
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
        else:
            X_test_scaled = X_test.values

        model.fit(X_train, y_train)

        # Prediction
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)

        # In sklearn/xgboost, predict might return 2D array, let's flatten
        if y_pred.ndim > 1:
            y_pred = y_pred.flatten()

        # Metrics
        # average='macro' calculates metrics for each label, and finds their unweighted mean.
        # This does not take label imbalance into account.
        precisions.append(precision_score(y_test, y_pred, average='macro', zero_division=0))
        f1s.append(f1_score(y_test, y_pred, average='macro', zero_division=0))

        # Multi-class PR-AUC (One-vs-Rest)
        try:
            # For PR-AUC, we use average_precision_score with macro average across classes
            # Note: requires scikit-learn >= 0.24 and probabilities
            # Convert y_test to one-hot for PR-AUC
            y_test_dummies = pd.get_dummies(y_test).values

            # Make sure shape matches (sometimes a class might not be present in test set)
            if y_test_dummies.shape[1] == y_pred_proba.shape[1]:
                prauc = average_precision_score(y_test_dummies, y_pred_proba, average='macro')
                praucs.append(prauc)
        except Exception as e:
            pass # Skip if error occurs due to missing classes

        # Inference Time test (1 row)
        single_row = X_test_scaled[[0]]
        start_time = time.perf_counter()
        _ = model.predict(single_row)
        end_time = time.perf_counter()
        inf_times.append((end_time - start_time) * 1000) # Convert to ms

    results.append({
        'Model': name,
        'Precision (Macro)': np.mean(precisions),
        'Macro F1-Score': np.mean(f1s),
        'PR-AUC': np.mean(praucs) if praucs else np.nan,
        'Inference Time (ms)': np.mean(inf_times)
    })

# Display Results
results_df = pd.DataFrame(results).set_index('Model')
results_df.sort_values('Precision (Macro)', ascending=False, inplace=True)
results_df